# DIPSER on Colab: get only what we need from 668 GB

Raw data is uncompressed PNG frames over FTP, far too big to keep. `src/dipser_fetch.py` never stores it:

| Stage | Cell | Downloads | Kept on Drive (`AffectProject/dipser/`) |
|---|---|---|---|
| 1. List | 4 | nothing | `manifest.csv`: every remote file + size |
| 2. Labels only | 5 | a few MB per zip (reads just the label + watch files out of each remote zip) | `labels/` |
| 3. Fetch selected zips | 7-9 | one zip at a time to `/content`, deleted after | `videos/`: per camera folder an `.mp4` + `.frames.csv`, plus `extras.tar.gz` |

CPU runtime is enough for all of it. Every stage resumes if rerun after a disconnect.

## 1. Settings

In [ ]:
REPO = "pauldsun/AffectProject"
DRIVE_OUT = "/content/drive/MyDrive/AffectProject"
DIPSER = f"{DRIVE_OUT}/dipser"          # manifest, labels, videos
WORK = "/content/dipser_work"           # local scratch for one zip at a time (not Drive)
TLS = ""                                # set to "--tls" if the plain FTP login is refused
CODE = "/content/affect"

## 2. Drive + code (pulls the latest push)

In [ ]:
import os, re, glob, json, shutil
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DIPSER, exist_ok=True)
if os.path.isdir(f"{CODE}/.git"):
    !git -C {CODE} pull --ff-only
else:
    !git clone -q https://github.com/{REPO}.git {CODE}
%cd {CODE}
!git log -1 --format="code at %h: %s (%cr)"

## 3. FTP credentials
From the 🔑 Secrets panel: `DIPSER_HOST`, `DIPSER_PORT`, `DIPSER_USER`, `DIPSER_PASS` (notebook access on).
**Never type them into a cell**: this notebook is in a public GitHub repo.

In [ ]:
from google.colab import userdata
for k in ("DIPSER_HOST", "DIPSER_PORT", "DIPSER_USER", "DIPSER_PASS"):
    os.environ[k] = userdata.get(k)
print("credentials loaded for host", os.environ["DIPSER_HOST"])

## 4. Stage 1: list the server (downloads nothing)
Prints sizes by file type and by group/experiment. Starts at `/DIPSEER` (the server also has a `V6` folder this
account may not open). Send this output.

In [ ]:
!python src/dipser_fetch.py list --out "{DIPSER}" {TLS} --root /DIPSEER --depth 3

## 5. Stage 2: labels only, for every zip
Reads the zip's table of contents and the `labels/` + `watch_sensor/` files over FTP, skipping the frames.
Room-camera zips (`context_cam_*.zip`) are skipped: labels are per student. Add `--include 'group_01/'` to try it on
part first. Resumes if rerun.

In [ ]:
!python src/dipser_fetch.py labels --out "{DIPSER}" {TLS} --exclude context_cam

In [ ]:
# what was found inside the zips + a look at one student's files (send this output)
import collections
per = {}
for d in sorted(glob.glob(f"{DIPSER}/labels/**/.done", recursive=True)):
    per[os.path.relpath(os.path.dirname(d), f"{DIPSER}/labels")] = json.load(open(d))
kinds = collections.Counter(("context_cam" if "context_cam" in r else "student",
                             "with files" if m else "nothing found") for r, m in per.items())
print(len(per), "zips read:", dict(kinds))
students = {r: m for r, m in per.items() if "context_cam" not in r}
nlab = collections.Counter(sum(bool(re.search(r"(^|/)labels?/", x)) for x in m) for m in students.values())
nwat = collections.Counter(sum(bool(re.search(r"(^|/)watch_sensors?/", x)) for x in m) for m in students.values())
print("label files per student zip -> number of zips:", dict(sorted(nlab.items())))
print("watch files per student zip -> number of zips:", dict(sorted(nwat.items())))
print("students with no label file:", [r for r, m in students.items() if not any(re.search(r"(^|/)labels?/", x) for x in m)][:10])
have = [r for r, m in students.items() if m]
if have:
    first = f"{DIPSER}/labels/{have[0]}"
    print("\n== files from", have[0])
    !find "{first}" -type f -not -name .done -exec ls -la {{}} \; | head -20
    for f in sorted(glob.glob(f"{first}/**/*.json", recursive=True))[:7]:
        txt = open(f).read()
        print(f"\n== {f.replace(first, '')}  ({len(txt) / 1e3:.0f} KB)"); print(txt[:800])

## 5b. Label analysis (no video needed; a few minutes)
Agreement between raters, boredom and boredom onsets per experiment, size of the "bored within 30 s" task, and
whether labels carry temporal signal. Saves `analysis/timeline.csv.gz` (per second, every rater), `recordings.csv`,
`experiments.csv`. `--k 2` = bored when at least 2 of the 4 experts say so. Send this output.

In [ ]:
!python src/dipser_labels.py "{DIPSER}/labels" --out "{DIPSER}/analysis" --k 2 --horizon 30

## 6. Choose what to fetch
`INCLUDE` / `EXCLUDE` are regular expressions on the remote path (see the manifest), e.g.
`INCLUDE = "group_01/experiment_0(1|3)/"`. Room cameras (`context_cam_*.zip`, ~1.8 GB each) are excluded by default;
set `EXCLUDE = ""` to include them. The dry run shows what would be fetched and its size.

In [ ]:
INCLUDE = ""        # fill in after looking at the manifest (cell 4) and labels (cell 5)
EXCLUDE = "context_cam"
sel = f'--include "{INCLUDE}"' if INCLUDE else ""
sel += f' --exclude "{EXCLUDE}"' if EXCLUDE else ""
!python src/dipser_fetch.py fetch --out "{DIPSER}" --work {WORK} {TLS} {sel} --dry-run
!df -h /content | tail -1

## 7. Stage 3, test: fetch ONE zip (measures download speed, checks the output)

In [ ]:
!python src/dipser_fetch.py fetch --out "{DIPSER}" --work {WORK} {TLS} {sel} --limit 1

In [ ]:
import pandas as pd, cv2
from IPython.display import Image, display
d = sorted(glob.glob(f"{DIPSER}/videos/**/done.json", recursive=True))[0]
print(open(d).read())
folder = os.path.dirname(d)
!ls -la "{folder}"
for v in sorted(glob.glob(f"{folder}/*.mp4")):
    !ffprobe -v error -select_streams v -show_entries stream=width,height,nb_frames,r_frame_rate -of csv=p=0 "{v}"
    print(pd.read_csv(v.replace(".mp4", ".frames.csv")).head(3))
    cap = cv2.VideoCapture(v); cap.set(cv2.CAP_PROP_POS_FRAMES, 100); ok, fr = cap.read()
    if ok:
        cv2.imwrite("/content/peek.jpg", fr); display(Image("/content/peek.jpg", width=480))

## 8. Stage 3: fetch the selection (hours; keep the tab open, rerun after a disconnect, it resumes)
Each line reports GB downloaded -> MB kept, and the download speed.

In [ ]:
!python src/dipser_fetch.py fetch --out "{DIPSER}" --work {WORK} {TLS} {sel}

## 9. Progress

In [ ]:
done = glob.glob(f"{DIPSER}/videos/**/done.json", recursive=True)
stats = [json.load(open(f)) for f in done]
print(f"{len(done)} zips done: {sum(s['zip_mb'] for s in stats) / 1e3:.1f} GB downloaded -> "
      f"{sum(v['mb'] for s in stats for v in s['videos'].values()) / 1e3:.2f} GB of video kept")
if stats:
    print("median download speed", pd.Series([s["download_mb_s"] for s in stats]).median(), "MB/s")
!du -sh "{DIPSER}"